# Sesion 8 — De Datos Crudos a Dataset Analizable
## Diplomado: Machine Learning en Seguros · FC UNAM
### 2 de mayo de 2026  ·  07:00 - 11:00 h  (4 horas)

---

> **Premisa de la sesion:** recibes los archivos crudos de la aseguradora.
> Tu trabajo: convertirlos en un dataset limpio, bien tipado y eficiente,
> resolviendo 7 problemas reales uno por uno.

---

**Prerequisito:** Debe existir la carpeta `datos/` con los archivos CSV.

## Las 7 Dudas que Resolvemos Hoy

| # | Duda | Herramienta |
|---|------|-------------|
| 1 | 46 columnas — ¿cuales necesito? | Taxonomia + `usecols` |
| 2 | Texto sucio: M/F/Masculino/Femenino | `str` operations |
| 3 | Fechas como texto: '15/04/2026' | `pd.to_datetime()` |
| 4 | API de reaseguro devuelve JSON | `read_json()` + `json_normalize()` |
| 5 | 90k filas, 11MB sin esperar | `chunks` + `category` |
| 6 | Downcast rompio precision de primas | Estrategia segura de `dtypes` |
| 7 | ¿Cuando cambiar a Polars? | `polars` — intro y comparativa |

---
## ACT 1 — El Dataset Crudo

### Duda 1: 46 columnas — ¿cuales necesito?

In [40]:
import pandas as pd
import numpy as np
import os, time

# ── Paso 1: Cargar TODO para entender que hay ────────────────────────────────
# Primera regla: antes de descartar, entende lo que tienes
df_todo = pd.read_csv('datos/cartera_polizas.csv', nrows=5)  # solo 5 filas para ver
print(f'Columnas totales: {len(df_todo.columns)}')
print()
print('Lista de columnas:')
for i, col in enumerate(df_todo.columns, 1):
    print(f'  {i:>2}. {col}')

Columnas totales: 46

Lista de columnas:
   1. id_poliza
   2. num_poliza
   3. id_contrato_interno
   4. folio_emision
   5. id_sistema_legacy
   6. nombre
   7. apellido_paterno
   8. apellido_materno
   9. nombre_completo
  10. rfc
  11. fecha_nacimiento
  12. edad
  13. sexo
  14. estado_civil
  15. ocupacion
  16. nivel_educacion
  17. ramo
  18. plan
  19. fecha_emision
  20. fecha_inicio_vigencia
  21. fecha_fin_vigencia
  22. num_renovaciones
  23. status_poliza
  24. motivo_baja
  25. canal_venta
  26. marca_vehiculo
  27. modelo_vehiculo
  28. tipo_vehiculo
  29. suma_asegurada
  30. deducible
  31. prima_neta
  32. prima_total
  33. cuota_prima
  34. forma_pago
  35. num_cuotas
  36. agente_id
  37. estado
  38. municipio
  39. codigo_postal
  40. coord_lat
  41. coord_lon
  42. version_documento
  43. hash_documento
  44. timestamp_carga
  45. usuario_captura
  46. ip_carga


In [41]:
# ── Paso 2: Diagnostico de todas las columnas ────────────────────────────────
# Cargamos todo para el diagnostico inicial
df_full = pd.read_csv('datos/cartera_polizas.csv')
mb_full = df_full.memory_usage(deep=True).sum() / 1024**2
print(f'Dataset completo: {df_full.shape} · {mb_full:.1f} MB')
print()

# Perfil de cada columna: tipo, NaN%, valores unicos
print(f'{"Columna":<30} {"Dtype":<12} {"NaN%":>7} {"Unicos":>8}')
print('-' * 65)
for col in df_full.columns:
    dtype = str(df_full[col].dtype)
    nan_pct = df_full[col].isna().mean() * 100
    n_uniq  = df_full[col].nunique()
    print(f'{col:<30} {dtype:<12} {nan_pct:>6.1f}% {n_uniq:>8,}')

Dataset completo: (50000, 46) · 112.0 MB

Columna                        Dtype           NaN%   Unicos
-----------------------------------------------------------------
id_poliza                      object          0.0%   50,000
num_poliza                     object          0.0%   50,000
id_contrato_interno            object          0.0%   48,572
folio_emision                  object          0.0%   49,870
id_sistema_legacy              object          0.0%   49,988
nombre                         object          0.0%       55
apellido_paterno               object          0.0%       38
apellido_materno               object          0.0%       23
nombre_completo                object          0.0%   31,140
rfc                            object          0.0%   50,000
fecha_nacimiento               object          0.0%   15,676
edad                           int64           0.0%       46
sexo                           object          0.0%        4
estado_civil                   object 

In [42]:
# ── Paso 3: Clasificar las columnas por categoria ────────────────────────────
# Esta clasificacion es una DECISION DE NEGOCIO — no solo tecnica

ANALITICAS = [
    'id_poliza','num_poliza','ramo','plan','status_poliza',
    'nombre','apellido_paterno','apellido_materno',
    'rfc','edad','sexo','estado_civil','ocupacion',
    'fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia', 'fecha_nacimiento',
    'num_renovaciones','motivo_baja',
    'suma_asegurada','deducible','prima_neta','prima_total',
    'forma_pago','agente_id','canal_venta',
    'estado','municipio','codigo_postal',
    'marca_vehiculo','modelo_vehiculo','tipo_vehiculo',  # solo Autos
]

REDUNDANTES = [
    'nombre_completo',        # = nombre + ap_pat + ap_mat
    'id_contrato_interno',    # ≈ id_poliza
    'folio_emision',          # ≈ num_poliza
    'cuota_prima',            # = prima_total / num_cuotas
    'num_cuotas',             # derivado de forma_pago
]

ADMINISTRATIVAS = [
    'hash_documento',         # hash SHA del PDF — auditoria IT
    'timestamp_carga',        # mismo valor para todos
    'ip_carga',               # IP del servidor batch
    'usuario_captura',        # operacion interna
    'version_documento',      # version del formato del contrato
    'id_sistema_legacy',      # util SOLO para joins con sistema core
]

CONDICIONALES = [
    'nivel_educacion',        # si lo necesitas para el modelo
    'coord_lat','coord_lon',  # solo si haces analisis geoespacial
]

print(f'Analiticas:     {len(ANALITICAS)}')
print(f'Redundantes:    {len(REDUNDANTES)}')
print(f'Administrativas:{len(ADMINISTRATIVAS)}')
print(f'Condicionales:  {len(CONDICIONALES)}')
print(f'Total:          {len(ANALITICAS)+len(REDUNDANTES)+len(ADMINISTRATIVAS)+len(CONDICIONALES)}')

Analiticas:     32
Redundantes:    5
Administrativas:6
Condicionales:  3
Total:          46


In [43]:
# ── Paso 4: Cargar SOLO las columnas que necesitamos ─────────────────────────
t0 = time.time()
df = pd.read_csv(
    'datos/cartera_polizas.csv',
    usecols=ANALITICAS,
    na_values=['N/D','N/A','ND','--','Sin dato',''],
)
t1 = time.time()
mb_opt = df.memory_usage(deep=True).sum() / 1024**2

print('=== COMPARATIVA DE CARGA ===')
print(f'Carga completa (46 cols):  {mb_full:.1f} MB')
print(f'Carga selectiva ({len(ANALITICAS)} cols):  {mb_opt:.1f} MB')
print(f'Reduccion de memoria:      {(1-mb_opt/mb_full)*100:.0f}%')
print(f'Tiempo de carga:           {(t1-t0)*1000:.0f} ms')
print()
print(f'Dataset de trabajo: {df.shape}')
df.head(3)

=== COMPARATIVA DE CARGA ===
Carga completa (46 cols):  112.0 MB
Carga selectiva (32 cols):  73.7 MB
Reduccion de memoria:      34%
Tiempo de carga:           343 ms

Dataset de trabajo: (50000, 32)


,id_poliza,num_poliza,nombre,apellido_paterno,apellido_materno,rfc,fecha_nacimiento,edad,sexo,estado_civil,...,tipo_vehiculo,suma_asegurada,deducible,prima_neta,prima_total,forma_pago,agente_id,estado,municipio,codigo_postal
0,POL-000001,Vid-21-000001,Gabriela,Moreno,Vega,MOGV020429CG6,29/04/2002,24,F,Union libre,...,NaN,3000000,NaN,54000.0,67651.20,Mensual,AG054,Veracruz,Poza Rica,36619.0
1,POL-000002,Aut-19-000002,Valeria,Torres,Castillo,TOVC020815IA8,15/08/2002,23,Femenino,Casado,...,Compacto,150000,8000.0,5250.0,6394.50,Trimestral,AG004,Michoacan,Morelia,58889.0
2,POL-000003,GMM-22-000003,Fernanda,Ramos,Silva,RAFS941018BC1,18/10/1994,31,M,Union libre,...,NaN,800000,5000.0,17600.0,22049.28,Mensual,AG051,Baja California,Tecate,45784.0


### 📝 Ejercicio 1 — Auditoria de columnas (8 min)

Usando el perfil que generaste arriba:
- **1a.** Identifica las 3 columnas con mas NaN — ¿tienen sentido esos NaN o son errores?
- **1b.** Encuentra columnas con menos de 5 valores unicos — ¿cuales deberian ser `category`?
- **1c.** Hay una columna numerica con un rango imposible (negativo o muy alto) — ¿cual es?
  *Pista: usa `df.describe()` sobre las columnas analiticas*

In [44]:
# Tu codigo aqui:

print("Top 3 columnas con mas valores faltantes:")
mas_faltantes = df.isna().sum().sort_values(ascending=False).head(3)
print(mas_faltantes)

print()
print("Columnas con menos de 5 valores únicos:")
menos5_unicos = df.nunique()
print(menos5_unicos[menos5_unicos < 5])


print()
print(df.describe().round(2))

df[df["prima_neta"] == df["prima_neta"].min()][["num_poliza", "ramo", "prima_neta", "prima_total"]]


Top 3 columnas con mas valores faltantes:
motivo_baja       47535
marca_vehiculo    35248
tipo_vehiculo     35248
dtype: int64

Columnas con menos de 5 valores únicos:
sexo             4
ramo             4
status_poliza    4
motivo_baja      4
forma_pago       4
dtype: int64

           edad  num_renovaciones  modelo_vehiculo  suma_asegurada  deducible  \
count  50000.00          50000.00         14752.00        50000.00   42490.00   
mean      43.32              1.24          2020.00      1002000.00    8266.82   
std       12.96              1.13             3.17      1105162.53    5503.03   
min       21.00              0.00          2015.00       150000.00    1000.00   
25%       32.00              0.00          2017.00       300000.00    5000.00   
50%       43.00              1.00          2020.00       500000.00    8000.00   
75%       54.00              2.00          2023.00      1000000.00   10000.00   
max       66.00              4.00          2025.00      5000000.00   20000.

,num_poliza,ramo,prima_neta,prima_total
56,Acc-21-000057,Accidentes Personales,2400.0,2784.00
61,Acc-19-000062,Accidentes Personales,2400.0,3006.72
70,Acc-21-000071,Accidentes Personales,2400.0,3006.72
86,Acc-22-000087,Accidentes Personales,2400.0,3006.72
117,Acc-19-000118,Accidentes Personales,2400.0,2784.00
...,...,...,...,...
49936,Acc-24-049937,Accidentes Personales,2400.0,2784.00
49945,Acc-23-049946,Accidentes Personales,2400.0,2867.52
49986,Acc-21-049987,Accidentes Personales,2400.0,2867.52
49993,Acc-22-049994,Accidentes Personales,2400.0,2784.00


---
### Duda 2: Texto Sucio — str Operations

El campo `sexo` tiene 4 representaciones distintas del mismo valor.
Si no lo normalizas, `groupby('sexo')` produce 8 grupos en lugar de 2.

In [45]:
# ── Ver el problema ──────────────────────────────────────────────────────────
print('Valores unicos en sexo ANTES de limpiar:')
print(df['sexo'].value_counts(dropna=False))
print(f'Total valores distintos: {df["sexo"].nunique()}')
print()

# Si haces groupby ahora, obtienes grupos incorrectos:
grupos_incorrectos = df.groupby('sexo')['prima_total'].mean()
print(grupos_incorrectos.to_string())
print(f'Grupos sin limpiar: {len(grupos_incorrectos)} (deberian ser 2)')

Valores unicos en sexo ANTES de limpiar:
sexo
M            16697
F            16585
Femenino      8466
Masculino     8252
Name: count, dtype: int64
Total valores distintos: 4

sexo
F            29053.277976
Femenino     29305.291745
M            29457.558936
Masculino    29369.674515
Grupos sin limpiar: 4 (deberian ser 2)


In [46]:
# ── Solucion: normalizar con str operations ──────────────────────────────────

# Paso 1: normalizar a mayusculas sin espacios
df['sexo'] = df['sexo'].str.strip().str.upper()

# Paso 2: mapear todas las variantes al estandar
MAPA_SEXO = {
    'M': 'M', 'MASCULINO': 'M', 'HOMBRE': 'M', 'MASC': 'M',
    'F': 'F', 'FEMENINO': 'F', 'MUJER': 'F', 'FEM': 'F',
}
df['sexo'] = df['sexo'].map(MAPA_SEXO)
# Valores no reconocidos quedan como NaN automaticamente

print('DESPUES de normalizar:')
print(df['sexo'].value_counts(dropna=False))
print()
# Ahora groupby correcto
print('Prima promedio por sexo:')
print(df.groupby('sexo')['prima_total'].mean().round(2))

DESPUES de normalizar:
sexo
F    25051
M    24949
Name: count, dtype: int64

Prima promedio por sexo:
sexo
F    29138.45
M    29428.49
Name: prima_total, dtype: float64


In [47]:
# ── Mas str operations sobre los datos reales ────────────────────────────────

# Limpiar codigo_postal: eliminar 'N/D' (ya convertido a NaN por na_values)
# Verificar:
print(f'CP con NaN: {df["codigo_postal"].isna().sum()}')
# Rellenar CP desconocido con 'DESCONOCIDO' para no perder la fila
df['codigo_postal'] = df['codigo_postal'].fillna('DESCONOCIDO')



CP con NaN: 1500


In [48]:
# Extraer ramo y anio desde num_poliza ('GMM-24-000123')
df['ramo_codigo']  = df['num_poliza'].str.extract(r'^([A-Z, a-z]+)-')
df['anio_poliza']  = df['num_poliza'].str.extract(r'-([0-9]{2})-').astype(float).astype('Int16')

# Verificar
print(df[['num_poliza','ramo_codigo','anio_poliza']].head(5).to_string(index=False))

   num_poliza ramo_codigo  anio_poliza
Vid-21-000001         Vid           21
Aut-19-000002         Aut           19
GMM-22-000003         GMM           22
Vid-19-000004         Vid           19
Vid-20-000005         Vid           20


---
### Duda 3: Fechas como Texto — pd.to_datetime()

In [49]:
# ── El problema: fechas llegaron como strings ────────────────────────────────
print('Tipo ANTES de convertir:', df['fecha_nacimiento'].dtype)
print('Muestra:', df['fecha_nacimiento'].head(3).values)
print()

# ── Convertir fecha_nacimiento (formato d/m/Y) ────────────────────────────────
df['fecha_nacimiento'] = pd.to_datetime(
    df['fecha_nacimiento'],
    format='%d/%m/%Y',
    errors='coerce'  # fechas invalidas → NaT (no detiene el proceso)
)

# ── Convertir fechas ISO estandar ────────────────────────────────────────────
for col_fecha in ['fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia']:
    df[col_fecha] = pd.to_datetime(df[col_fecha], errors='coerce')

print('Fechas convertidas:')
print(df[['fecha_nacimiento','fecha_emision','fecha_fin_vigencia']].dtypes)
print(f'NaT en fecha_nacimiento: {df["fecha_nacimiento"].isna().sum()}')

Tipo ANTES de convertir: object
Muestra: ['29/04/2002' '15/08/2002' '18/10/1994']

Fechas convertidas:
fecha_nacimiento      datetime64[ns]
fecha_emision         datetime64[ns]
fecha_fin_vigencia    datetime64[ns]
dtype: object
NaT en fecha_nacimiento: 0


In [50]:
# ── Calculos actuariales con las fechas convertidas ──────────────────────────
hoy = pd.Timestamp.today()

# Edad calculada (mas precisa que la columna 'edad' del CSV)
df['edad_calc'] = ((hoy - df['fecha_nacimiento']).dt.days / 365.25)

# Dias de vigencia de la poliza
df['dias_vigencia'] = (df['fecha_fin_vigencia'] - df['fecha_inicio_vigencia']).dt.days

# Fraccion expuesta (cuanto del periodo ya transcurrio)
dias_transcurridos = (hoy - df['fecha_inicio_vigencia']).dt.days
df['fraccion_expuesta'] = (dias_transcurridos / df['dias_vigencia']).clip(0, 1).round(4)

# Componentes de fecha para groupby temporal
df['anio_emision']     = df['fecha_emision'].dt.year
df['mes_emision']      = df['fecha_emision'].dt.month
df['trimestre_emision']= df['fecha_emision'].dt.quarter

# Verificar
print(df[['nombre','edad','edad_calc','dias_vigencia','fraccion_expuesta']].head(5).to_string(index=False))

  nombre  edad  edad_calc  dias_vigencia  fraccion_expuesta
Gabriela    24  24.016427            365                1.0
 Valeria    23  23.720739            366                1.0
Fernanda    31  31.545517            365                1.0
  Silvia    40  40.246407            366                1.0
 Antonio    29  29.785079            365                1.0


### 📝 Ejercicio 2 — Limpiar siniestros.csv (10 min)

El archivo `siniestros.csv` tiene fechas en 3 formatos distintos:
- `fecha_ocurrencia`: YYYY-MM-DD
- `fecha_apertura`: d/m/Y (mismo dia que fecha_reporte pero formato distinto — es REDUNDANTE)
- `fecha_ultimo_movimiento`: d/m/Y

**2a.** Carga siniestros.csv usando SOLO las columnas utiles (descarta las administrativas y redundantes).
**2b.** Convierte las 3 columnas de fecha a datetime con el formato correcto.
**2c.** Calcula `dias_reporte_real` = fecha_reporte - fecha_ocurrencia.
**2d.** Calcula `dias_resolucion_real` = fecha_cierre - fecha_reporte (NaT si no esta cerrado).
**2e.** Verifica: ¿cuantos siniestros llevan mas de 1100 dias sin cerrar?

In [51]:
# Tu codigo aqui:
siniestros_cols_utiles = [
    'id_siniestro','id_poliza','ramo','tipo_siniestro',
    'fecha_ocurrencia','fecha_reporte','fecha_ultimo_movimiento','fecha_cierre',
    'dias_reporte','monto_reclamado','monto_pagado',
    'status_siniestro','motivo_rechazo','id_ajustador',
]
# Carga, convierte fechas y calcula los campos derivados:
siniestros = pd.read_csv("datos/siniestros.csv", 
                         usecols=siniestros_cols_utiles, 
                         na_values=['N/D','N/A','ND','--','Sin dato',''])

cols_fechas = ['fecha_ocurrencia','fecha_reporte','fecha_cierre']
cols_fechas_todas = cols_fechas.copy()
cols_fechas_todas.append("fecha_ultimo_movimiento")

print('Fechas sin convertir:')
print(siniestros[cols_fechas_todas].head(3).to_string(index=False))
print()
print("Na en fechas antes de convertir:")
for col in cols_fechas_todas:
    print(f"{col}: {siniestros[col].isna().sum()}")
siniestros["fecha_ultimo_movimiento"] = pd.to_datetime(siniestros["fecha_ultimo_movimiento"], format='%d/%m/%Y', errors='coerce')

print("-"*100)
print()
for col in cols_fechas:
    siniestros[col] = pd.to_datetime(siniestros[col], errors='coerce')
print('Fechas convertidas:')
print(siniestros[cols_fechas_todas].head(3).to_string(index=False))
print()
print("Na en fechas DESPUES de convertir:")
for col in cols_fechas_todas:
    print(f"{col}: {siniestros[col].isna().sum()}")

siniestros["dias_reporte_real"] = (siniestros['fecha_reporte'] - siniestros['fecha_ocurrencia']).dt.days
siniestros["dias_resolución_real"] = (siniestros['fecha_cierre'] - siniestros['fecha_reporte']).dt.days

print("-"*100)
print("Siniestros con mas de 180 dias sin cerrar")
print(siniestros[siniestros["dias_resolución_real"] > 180]
      [['id_siniestro','id_poliza','fecha_ocurrencia','fecha_reporte','fecha_cierre','dias_reporte_real', 'dias_resolución_real']]
      .sort_values('dias_resolución_real', ascending=False).head(3).to_string(index=False))

Fechas sin convertir:
fecha_ocurrencia fecha_reporte fecha_cierre fecha_ultimo_movimiento
      2022-06-22    2022-06-25          NaN              11/08/2022
      2023-04-07    2023-04-14          NaN              11/06/2023
      2022-05-24    2022-05-26          NaN              16/06/2022

Na en fechas antes de convertir:
fecha_ocurrencia: 0
fecha_reporte: 0
fecha_cierre: 7169
fecha_ultimo_movimiento: 0
----------------------------------------------------------------------------------------------------

Fechas convertidas:
fecha_ocurrencia fecha_reporte fecha_cierre fecha_ultimo_movimiento
      2022-06-22    2022-06-25          NaT              2022-08-11
      2023-04-07    2023-04-14          NaT              2023-06-11
      2022-05-24    2022-05-26          NaT              2022-06-16

Na en fechas DESPUES de convertir:
fecha_ocurrencia: 0
fecha_reporte: 0
fecha_cierre: 7169
fecha_ultimo_movimiento: 0
----------------------------------------------------------------------------

---
### Duda 4: JSON — Datos de API


In [52]:
# ── Simular una respuesta JSON de una API de reaseguro ───────────────────────
import json

# Este es el tipo de JSON que recibirias de una API REST
respuesta_api = {
    'timestamp': '2026-05-02T07:00:00',
    'origen': 'Sistema_Reaseguro_v3.1',
    'polizas_reaseguradas': [
        {'id_poliza':'POL-000001','ramo':'GMM',
         'reasegurador':{'nombre':'Munich Re','participacion':0.40,'prima':960.0},
         'limites':{'maximo':2_000_000,'retencion':500_000}},
        {'id_poliza':'POL-000002','ramo':'Vida',
         'reasegurador':{'nombre':'Swiss Re','participacion':0.35,'prima':1820.0},
         'limites':{'maximo':5_000_000,'retencion':1_000_000}},
        {'id_poliza':'POL-000005','ramo':'GMM',
         'reasegurador':{'nombre':'Munich Re','participacion':0.40,'prima':550.0},
         'limites':{'maximo':2_000_000,'retencion':500_000}},
    ]
}

# Guardar como JSON (simula lo que llegaria de la API)
with open('datos/respuesta_reaseguro.json','w',encoding='utf-8') as f:
    json.dump(respuesta_api, f, ensure_ascii=False, indent=2)

print('JSON guardado en datos/respuesta_reaseguro.json')
print('Primeras lineas:')
print(json.dumps(respuesta_api, indent=2, ensure_ascii=False)[:300])

JSON guardado en datos/respuesta_reaseguro.json
Primeras lineas:
{
  "timestamp": "2026-05-02T07:00:00",
  "origen": "Sistema_Reaseguro_v3.1",
  "polizas_reaseguradas": [
    {
      "id_poliza": "POL-000001",
      "ramo": "GMM",
      "reasegurador": {
        "nombre": "Munich Re",
        "participacion": 0.4,
        "prima": 960.0
      },
      "limites": 


In [53]:
# ── Problema: JSON anidado no se carga directo en DataFrame ─────────────────
# pd.read_json funciona para JSON simple pero no para estructuras anidadas

# Cargar el JSON
with open('datos/respuesta_reaseguro.json') as f:
    data = json.load(f)

# Intentar con pd.read_json — no da el resultado esperado con anidados
# pd.read_json('datos/respuesta_reaseguro.json')  # solo lee nivel 1

# ── Solucion: json_normalize aplana el JSON anidado ──────────────────────────
from pandas import json_normalize

df_reas = json_normalize(
    data['polizas_reaseguradas'],
    sep='_'    # separador para campos anidados
)

print('DataFrame aplanado:')
print(df_reas.to_string(index=False))
print()
print('Columnas generadas:', list(df_reas.columns))

DataFrame aplanado:
 id_poliza ramo reasegurador_nombre  reasegurador_participacion  reasegurador_prima  limites_maximo  limites_retencion
POL-000001  GMM           Munich Re                        0.40               960.0         2000000             500000
POL-000002 Vida            Swiss Re                        0.35              1820.0         5000000            1000000
POL-000005  GMM           Munich Re                        0.40               550.0         2000000             500000

Columnas generadas: ['id_poliza', 'ramo', 'reasegurador_nombre', 'reasegurador_participacion', 'reasegurador_prima', 'limites_maximo', 'limites_retencion']


In [54]:
# ── json_normalize con listas anidadas ───────────────────────────────────────
# Caso mas complejo: cuando hay listas dentro del JSON

respuesta_multi = {
    'polizas': [
        {'id':'P01','coberturas':[{'tipo':'Hospitalizacion','suma':500_000},{'tipo':'Cirugia','suma':300_000}]},
        {'id':'P02','coberturas':[{'tipo':'Hospitalizacion','suma':1000_000}]},
    ]
}

# record_path: donde esta la lista a 'explotar'
# meta: campos del nivel padre que quieres conservar
df_cob = json_normalize(
    respuesta_multi['polizas'],
    record_path='coberturas',
    meta=['id'],
    sep='_'
)
print(df_cob)

              tipo     suma   id
0  Hospitalizacion   500000  P01
1          Cirugia   300000  P01
2  Hospitalizacion  1000000  P02


In [55]:
# ── Guardar DataFrame como JSON ──────────────────────────────────────────────

# orient='records' — lista de objetos (lo mas comun para APIs)
df.head(10).to_json('datos/muestra.json',
    orient='records',
    force_ascii=False,  # preserva caracteres especiales (acentos)
    indent=2,
    date_format='iso'   # fechas en formato ISO
)

# Verificar
with open('datos/muestra.json') as f:
    preview = f.read(300)
print(preview)

[
  {
    "id_poliza":"POL-000001",
    "num_poliza":"Vid-21-000001",
    "nombre":"Gabriela",
    "apellido_paterno":"Moreno",
    "apellido_materno":"Vega",
    "rfc":"MOGV020429CG6",
    "fecha_nacimiento":"2002-04-29T00:00:00.000",
    "edad":24,
    "sexo":"F",
    "estado_civil":"Union libre",


---
### Duda 5: 90k Filas — Procesar sin Esperar (chunks)

In [56]:
# ── Ver el tamano del archivo de beneficiarios ───────────────────────────────
mb_ben = os.path.getsize('datos/beneficiarios.csv') / 1024**2
print(f'beneficiarios.csv: {mb_ben:.1f} MB')

# Contar filas sin cargar todo
total_ben = sum(len(chunk) for chunk in
    pd.read_csv('datos/beneficiarios.csv', chunksize=10_000))
print(f'Total beneficiarios: {total_ben:,}')

# ── Patron real: calcular estadisticas por parentesco ─────────────────────────
# Sin cargar los 90k en memoria a la vez
conteo_parentesco = {}

for chunk in pd.read_csv('datos/beneficiarios.csv', chunksize=10_000):
    counts = chunk['parentesco'].value_counts().to_dict()
    for k, v in counts.items():
        conteo_parentesco[k] = conteo_parentesco.get(k, 0) + v

resultado = pd.Series(conteo_parentesco).sort_values(ascending=False)
print('Beneficiarios por parentesco (procesado por chunks):')
print(resultado)

beneficiarios.csv: 11.5 MB
Total beneficiarios: 90,000
Beneficiarios por parentesco (procesado por chunks):
Conyuge    26718
Hijo       22785
Hija       17871
Madre       7298
Padre       7173
Hermano     3596
Hermana     2765
Otro        1794
dtype: int64


In [57]:
# ── Filtrar y concatenar solo lo que necesitas ───────────────────────────────
# Obtener solo beneficiarios activos de polizas de Vida

partes = []
for chunk in pd.read_csv('datos/beneficiarios.csv',
                         chunksize=10_000,
                         usecols=['id_beneficiario','id_poliza','nombre',
                                  'apellido_paterno','parentesco','porcentaje','activo']):
    filtrado = chunk[chunk['activo'] == True]
    if len(filtrado) > 0:
        partes.append(filtrado)

ben_activos = pd.concat(partes, ignore_index=True)
print(f'Beneficiarios activos: {len(ben_activos):,}')
print(f'Memoria: {ben_activos.memory_usage(deep=True).sum()/(1024):.0f} KB')

Beneficiarios activos: 67,656
Memoria: 21956 KB


---
### Duda 6: Downcast — Cuándo Es Seguro

In [58]:
# ── Demostrar el problema de precision ───────────────────────────────────────
import numpy as np

# float64 vs float32 con valores grandes
prima_grande = 15_432_756.100
print(f'Original (float64): {prima_grande}')
print(f'Como float32:       {np.float32(prima_grande)}')
print(f'Diferencia:         {abs(prima_grande - float(np.float32(prima_grande))):.2f}')
print()

# Con valores tipicos de primas individuales
prima_gmm = 3_450.75
print(f'Prima GMM (float64): {prima_gmm}')
print(f'Prima GMM (float32): {np.float32(prima_gmm)}')
print(f'Diferencia:          {abs(prima_gmm - float(np.float32(prima_gmm))):.6f}')

Original (float64): 15432756.1
Como float32:       15432756.0
Diferencia:         0.10

Prima GMM (float64): 3450.75
Prima GMM (float32): 3450.75
Diferencia:          0.000000


In [59]:
# ── Estrategia segura de optimizacion ────────────────────────────────────────
print('Memoria ANTES de optimizar:')
print(f'{df.memory_usage(deep=True).sum()/1024**2:.2f} MB')
print()

df_opt = df.copy()

# SEGURO: categoricas para columnas con pocos valores unicos
cols_category = ['ramo','plan','status_poliza','sexo','canal_venta',
                 'forma_pago','estado','estado_civil','tipo_vehiculo']
for col in cols_category:
    if col in df_opt.columns:
        n_uniq = df_opt[col].nunique()
        n_tot  = len(df_opt)
        pct = n_uniq/n_tot
        print(f'  {col:<25}: {n_uniq:>5} unicos ({pct:.1%}) → category')
        df_opt[col] = df_opt[col].astype('category')

# SEGURO: enteros pequenos
df_opt['num_renovaciones'] = df_opt['num_renovaciones'].fillna(0).astype('int8')

# SEGURO: booleano
# (activa ya no esta porque la descartamos, pero aplica el principio)

# CONSERVAR en float64: primas y sumas (montos grandes o con centavos importantes)
# NO hacer: df_opt['prima_total'] = df_opt['prima_total'].astype('float32')

print()
print('Memoria DESPUES de optimizar:')
mb_antes = df.memory_usage(deep=True).sum()/1024**2
mb_desp  = df_opt.memory_usage(deep=True).sum()/1024**2
print(f'{mb_antes:.2f} MB → {mb_desp:.2f} MB ({(1-mb_desp/mb_antes)*100:.0f}% reduccion)')

Memoria ANTES de optimizar:
68.25 MB

  ramo                     :     4 unicos (0.0%) → category
  plan                     :    12 unicos (0.0%) → category
  status_poliza            :     4 unicos (0.0%) → category
  sexo                     :     2 unicos (0.0%) → category
  canal_venta              :     6 unicos (0.0%) → category
  forma_pago               :     4 unicos (0.0%) → category
  estado                   :    15 unicos (0.0%) → category
  estado_civil             :     5 unicos (0.0%) → category
  tipo_vehiculo            :     8 unicos (0.0%) → category

Memoria DESPUES de optimizar:
68.25 MB → 42.18 MB (38% reduccion)


### 📝 Ejercicio 3 — Pipeline de limpieza completo (12 min)

Encapsula todo lo que hicimos en una funcion `limpiar_cartera(ruta_csv)` que:
- Carga con `usecols=ANALITICAS`
- Normaliza `sexo` con str + map
- Convierte fechas con `to_datetime` + `errors='coerce'`
- Calcula `edad_calc`, `dias_vigencia`, `fraccion_expuesta`
- Aplica optimizacion de memoria (categoricas)
- Retorna el DataFrame limpio

Al final llama: `df_limpio = limpiar_cartera('datos/cartera_polizas.csv')`
y verifica que no tenga texto sucio en sexo.

In [60]:
# Tu codigo aqui:
def limpiar_cartera(ruta_csv, columnas_analiticas=ANALITICAS):
    # Cargar solo las columnas analiticas en partes para optimizar memoria
    partes = []
    for chunk in pd.read_csv(ruta_csv,
                             usecols=columnas_analiticas,
                             na_values=['N/D','N/A','ND','--','Sin dato',''],
                             chunksize=10_000):
        if len(chunk) > 0:
            partes.append(chunk)
    datos = pd.concat(partes, ignore_index=True)
    
    # Normalizar sexo
    MAPA_SEXO = {
    'M': 'M', 'MASCULINO': 'M', 'HOMBRE': 'M', 'MASC': 'M',
    'F': 'F', 'FEMENINO': 'F', 'MUJER': 'F', 'FEM': 'F',
    }
    datos['sexo'] = datos['sexo'].str.strip().str.upper().map(MAPA_SEXO) # Nomalizamos la columna sexo

    # Convertir fechas
    cols_fechas = ['fecha_emision', 'fecha_inicio_vigencia', 'fecha_fin_vigencia']
    for col in cols_fechas:
        datos[col] = pd.to_datetime(datos[col], errors='coerce') # Convertimos las columnas de fecha a formato datetime
    datos["fecha_nacimiento"] = pd.to_datetime(datos["fecha_nacimiento"], format="%d/%m/%Y", errors='coerce') # Convertimos la columna de fecha de nacimiento a formato datetime
    
    # Calculos actuariales
    hoy = pd.Timestamp.today()
    datos["edad_calc"] = ((hoy - datos['fecha_nacimiento']).dt.days / 365.25) # Calculamos la edad a partir de la fecha de nacimiento
    datos['dias_vigencia'] = (datos['fecha_fin_vigencia'] - datos['fecha_inicio_vigencia']).dt.days # Calculamos los dias de vigencia de la poliza
    datos["fraccion_expuesta"] = ( (hoy - datos['fecha_inicio_vigencia']).dt.days / datos['dias_vigencia'] ).clip(0, 1).round(4) # Calculamos la fraccion expuesta

    # Optimiazar tipos de datos
    cols_category = ['ramo','plan','status_poliza','sexo','canal_venta',
                 'forma_pago','estado','estado_civil','tipo_vehiculo']
    for col in cols_category:
            datos[col] = datos[col].astype('category') # Convertimos las columnas categoricas a tipo category para optimizar memoria
    
    return datos

# Correción: Deberiamos crear subfunciones para cada tipo de limpieza (limpiar_sexo, limpiar_fechas, etc) y luego llamarlas desde limpiar_cartera. 
# Esto mejora la legibilidad y mantenibilidad del código.

In [61]:
df = limpiar_cartera('datos/cartera_polizas.csv')
df.head(3)

,id_poliza,num_poliza,nombre,apellido_paterno,apellido_materno,rfc,fecha_nacimiento,edad,sexo,estado_civil,...,prima_neta,prima_total,forma_pago,agente_id,estado,municipio,codigo_postal,edad_calc,dias_vigencia,fraccion_expuesta
0,POL-000001,Vid-21-000001,Gabriela,Moreno,Vega,MOGV020429CG6,2002-04-29,24,F,Union libre,...,54000.0,67651.20,Mensual,AG054,Veracruz,Poza Rica,36619.0,24.016427,365,1.0
1,POL-000002,Aut-19-000002,Valeria,Torres,Castillo,TOVC020815IA8,2002-08-15,23,F,Casado,...,5250.0,6394.50,Trimestral,AG004,Michoacan,Morelia,58889.0,23.720739,366,1.0
2,POL-000003,GMM-22-000003,Fernanda,Ramos,Silva,RAFS941018BC1,1994-10-18,31,M,Union libre,...,17600.0,22049.28,Mensual,AG051,Baja California,Tecate,45784.0,31.545517,365,1.0


---
### Duda 7: ¿Cuándo Cambiar a Polars?

In [62]:
!pip install polars

In [63]:
# ── Instalar polars si no esta disponible ────────────────────────────────────
# En tu terminal: pip install polars
# Verificar:
try:
    import polars as pl
    print(f'Polars {pl.__version__} disponible')
    POLARS_OK = True
except ImportError:
    print('Polars no instalado. Ejecuta: pip install polars')
    POLARS_OK = False

Polars 1.40.1 disponible


In [64]:
# ── Comparativa de velocidad pandas vs polars ────────────────────────────────
if POLARS_OK:
    import polars as pl
    import time

    # ── Pandas ──────────────────────────────────────────────────────────────
    t0 = time.time()
    df_pd = pd.read_csv('datos/cartera_polizas.csv', usecols=ANALITICAS)
    res_pd = (df_pd.groupby('ramo')
                   .agg(polizas=('id_poliza','count'),
                        prima_total=('prima_total','sum'),
                        prima_prom=('prima_neta','mean'))
                   .round(2).reset_index())
    t_pd = time.time()-t0

    # ── Polars ──────────────────────────────────────────────────────────────
    t0 = time.time()
    df_pl = pl.read_csv('datos/cartera_polizas.csv', columns=ANALITICAS)
    res_pl = (df_pl
        .group_by('ramo')
        .agg([
            pl.col('id_poliza').count().alias('polizas'),
            pl.col('prima_total').sum().alias('prima_total'),
            pl.col('prima_neta').mean().alias('prima_prom'),
        ])
    )
    t_pl = time.time()-t0

    print(f'Pandas:  {t_pd*1000:.0f} ms')
    print(f'Polars:  {t_pl*1000:.0f} ms')
    print(f'Polars es {t_pd/t_pl:.1f}x mas rapido en esta operacion')
    print()
    print('Resultado Polars:')
    print(res_pl)
else:
    print('Instala polars para ver la comparativa: pip install polars')

Pandas:  387 ms
Polars:  23 ms
Polars es 17.1x mas rapido en esta operacion

Resultado Polars:
shape: (4, 4)
┌───────────────────────┬─────────┬─────────────┬──────────────┐
│ ramo                  ┆ polizas ┆ prima_total ┆ prima_prom   │
│ ---                   ┆ ---     ┆ ---         ┆ ---          │
│ str                   ┆ u32     ┆ f64         ┆ f64          │
╞═══════════════════════╪═════════╪═════════════╪══════════════╡
│ Vida                  ┆ 7510    ┆ 4.5433e8    ┆ 50439.777113 │
│ Autos                 ┆ 14752   ┆ 2.5382e8    ┆ 14349.345658 │
│ Accidentes Personales ┆ 5207    ┆ 2.4981e7    ┆ 4002.032838  │
│ GMM                   ┆ 22531   ┆ 7.3102e8    ┆ 27065.47908  │
└───────────────────────┴─────────┴─────────────┴──────────────┘


In [65]:
# ── Sintaxis Polars: las operaciones mas comunes ──────────────────────────────
if POLARS_OK:
    df_pl = pl.read_csv('datos/cartera_polizas.csv',
                        columns=['id_poliza','ramo','prima_total','edad'])

    # Filtrar
    print('Polizas con prima > 10,000:')
    print(df_pl.filter(pl.col('prima_total') > 10_000).shape)

    # Agregar columna
    df_pl2 = df_pl.with_columns(
        (pl.col('prima_total')/12).alias('prima_mensual')
    )

    # Sort
    print('Top 5 primas:')
    print(df_pl2.sort('prima_total', descending=True).head(5)[['id_poliza','ramo','prima_total']])

    # Lazy evaluation — ejecuta todo optimizado al final
    resultado = (
        pl.scan_csv('datos/cartera_polizas.csv')  # NO carga en memoria aun
        .filter(pl.col('prima_total') > 5000)
        .group_by('ramo')
        .agg(pl.col('prima_total').sum())
        .collect()  # AHORA ejecuta con el plan optimizado
    )
    print('Lazy evaluation result:')
    print(resultado)

Polizas con prima > 10,000:
(33919, 4)
Top 5 primas:
shape: (5, 3)
┌────────────┬──────┬─────────────┐
│ id_poliza  ┆ ramo ┆ prima_total │
│ ---        ┆ ---  ┆ ---         │
│ str        ┆ str  ┆ f64         │
╞════════════╪══════╪═════════════╡
│ POL-007740 ┆ Vida ┆ 182658.24   │
│ POL-007890 ┆ Vida ┆ 182658.24   │
│ POL-024748 ┆ Vida ┆ 182658.24   │
│ POL-003193 ┆ Vida ┆ 180403.2    │
│ POL-007981 ┆ Vida ┆ 180403.2    │
└────────────┴──────┴─────────────┘
Lazy evaluation result:
shape: (4, 2)
┌───────────────────────┬─────────────┐
│ ramo                  ┆ prima_total │
│ ---                   ┆ ---         │
│ str                   ┆ f64         │
╞═══════════════════════╪═════════════╡
│ GMM                   ┆ 7.3102e8    │
│ Accidentes Personales ┆ 1.2546e7    │
│ Vida                  ┆ 4.5433e8    │
│ Autos                 ┆ 2.5382e8    │
└───────────────────────┴─────────────┘


### Regla practica: ¿Pandas o Polars?

| Situacion | Usa |
|-----------|-----|
| Aprendizaje, primeros modelos | **pandas** — el ecosistema es enorme |
| < 1 millon de filas | **pandas** — mas que suficiente |
| Analisis interactivo en notebook | **pandas** — syntax mas conocida |
| Pipeline de produccion > 5M filas | **polars** — 5-20x mas rapido |
| Ingesta diaria de datos grandes | **polars** con lazy evaluation |
| ETL de empresa | **polars** o **spark** segun el tamano |

---
## pivot_table con Datos Reales


In [66]:
# ── Agregar columnas necesarias para el pivot ────────────────────────────────
df_work = pd.read_csv('datos/cartera_polizas.csv', usecols=ANALITICAS,
                      na_values=['N/D','N/A',''])
df_work['prima_neta'] = df_work['prima_neta'].fillna(df_work.groupby('ramo')['prima_neta'].transform('median'))
df_work['g_edad'] = pd.cut(df_work['edad'], bins=[0,30,45,60,100],
                           labels=['18-30','31-45','46-60','61+'])
df_work['siniest_flag'] = (df_work['prima_neta'] > df_work['prima_neta'].quantile(0.75)).astype(int)

print(f'Dataset para pivot: {df_work.shape}')

Dataset para pivot: (50000, 34)


In [67]:
# ── pivot_table: prima por ramo x grupo de edad ──────────────────────────────
tabla_prima = pd.pivot_table(
    df_work,
    values   = 'prima_total',
    index    = 'ramo',
    columns  = 'g_edad',
    aggfunc  = 'sum',
    fill_value = 0,
    margins    = True,
    margins_name = 'TOTAL'
).round(0) / 1_000  # en miles de pesos

print('Prima total por ramo y grupo de edad (miles MXN):')
print(tabla_prima.to_string())

Prima total por ramo y grupo de edad (miles MXN):
g_edad                      18-30       31-45       46-60         61+        TOTAL
ramo                                                                              
Accidentes Personales    5248.369    8348.395    8731.668    2652.247    24980.679
Autos                   55457.022   84225.512   84571.404   29569.569   253823.506
GMM                    139278.258  223908.688  264218.298  103617.078   731022.322
Vida                    79667.849  134774.641  166301.773   73587.860   454332.123
TOTAL                  279651.498  451257.236  523823.143  209426.754  1464158.631


C:\Users\gusta\AppData\Local\Temp\ipykernel_21580\3938379720.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  tabla_prima = pd.pivot_table(


In [68]:
# ── pivot_table: polizas por zona x canal ────────────────────────────────────
tabla_canal = pd.pivot_table(
    df_work,
    values   = 'id_poliza',
    index    = 'estado',
    columns  = 'canal_venta',
    aggfunc  = 'count',
    fill_value = 0,
    margins    = True,
    margins_name = 'TOTAL'
)

print('Polizas por estado y canal de venta:')
print(tabla_canal.to_string())

Polizas por estado y canal de venta:
canal_venta       Agente  Banca Seguros  Broker  Digital  Directo  Promotor  TOTAL
estado                                                                            
Baja California     1646            329     678      241      356        98   3348
CDMX                1719            311     684      217      318       108   3357
Chihuahua           1656            344     642      250      357       101   3350
Coahuila            1721            349     672      199      329        96   3366
Estado de Mexico    1703            336     652      253      287        93   3324
Guanajuato          1745            339     669      237      359        96   3445
Jalisco             1664            322     670      258      332       106   3352
Michoacan           1621            327     711      217      319        99   3294
Nuevo Leon          1681            335     615      225      326        96   3278
Puebla              1586            330     672   

---
## Exportar — CSV, Excel, Parquet y JSON


In [69]:
# ── Guardar en todos los formatos y comparar ──────────────────────────────────
df_export = df_work.head(10_000)  # subconjunto para demo

formatos = {
    'CSV':     ('datos/export_demo.csv',
                lambda: df_export.to_csv('datos/export_demo.csv', index=False)),
    'Parquet': ('datos/export_demo.parquet',
                lambda: df_export.to_parquet('datos/export_demo.parquet', index=False)),
    'JSON':    ('datos/export_demo.json',
                lambda: df_export.to_json('datos/export_demo.json',
                                          orient='records', force_ascii=False)),
}

print(f'{"Formato":<10} {"Tamano":>10} {"Tiempo":>10}')
print('-' * 35)
for nombre, (ruta, guardar) in formatos.items():
    t0 = time.time()
    guardar()
    t = (time.time()-t0)*1000
    kb = os.path.getsize(ruta)/1024
    print(f'{nombre:<10} {kb:>8.0f} KB {t:>8.0f} ms')

Formato        Tamano     Tiempo
-----------------------------------
CSV            2465 KB      149 ms
Parquet         633 KB       63 ms
JSON           7645 KB       71 ms


In [70]:
# ── Excel multihoja — el entregable mas solicitado en empresas ───────────────
resumen_ramo = df_work.groupby('ramo').agg(
    polizas    =('id_poliza','count'),
    prima_total=('prima_total','sum'),
    prima_prom =('prima_total','mean'),
).round(2).reset_index()
resumen_ramo['pct_cartera'] = (resumen_ramo['prima_total']/resumen_ramo['prima_total'].sum()*100).round(1)

with pd.ExcelWriter('datos/reporte_demo.xlsx', engine='openpyxl') as writer:
    df_work.head(5_000).to_excel(writer, sheet_name='Cartera', index=False)
    resumen_ramo.to_excel(writer, sheet_name='Resumen_Ramo', index=False)
    tabla_prima.to_excel(writer, sheet_name='Pivot_Prima')
    tabla_canal.to_excel(writer, sheet_name='Pivot_Canal')

kb_xl = os.path.getsize('datos/reporte_demo.xlsx')/1024
print(f'Excel multihoja generado: {kb_xl:.0f} KB con 4 hojas')

Excel multihoja generado: 972 KB con 4 hojas


---
## 🏆 Ejercicio Integrador Final — Pipeline Completo

**Contexto:** Tu jefa de estadistica te pide el reporte ejecutivo del Q1 2026.
Tienes los 4 archivos crudos. Debes construir un pipeline completo,
documentando cada decision de limpieza.

**Tiempo:** 40 minutos  |  **Entregable:** Excel con 5 hojas + Parquet

---

### Criterios de evaluacion:
- ✅ Cada decision de descarte de columnas esta justificada con comentario
- ✅ No hay texto sucio en columnas categoricas clave
- ✅ Todas las fechas son datetime, no object
- ✅ dtypes optimizados sin perder precision en montos
- ✅ El Excel tiene las 5 hojas con los datos correctos
- ✅ El Parquet es mas pequeno que el CSV equivalente

### Resolución del ejercicio:

In [71]:
# Haremos funciones para cada una de las tareas del proceso de limpieza, para que el codigo sea mas modular y facil de mantener.

def cargar_datos(ruta_csv, columnas_analiticas=None, chunksize=10_000):
    """Carga el CSV en partes, solo con las columnas analiticas, y concatena los resultados.
    
    Args:
        ruta_csv (str): Ruta al archivo CSV.
        columnas_analiticas (list): Lista de columnas a cargar.
        chunksize (int): Tamaño de los chunks para cargar el archivo en partes.
    
    Returns:
        pd.DataFrame: DataFrame con los datos cargados y concatenados.
    
    """
    partes = []
    for chunk in pd.read_csv(ruta_csv,
                             usecols=columnas_analiticas,
                             na_values=['N/D','N/A','ND','--','Sin dato',''],
                             chunksize=chunksize):
        if len(chunk) > 0:
            partes.append(chunk)
    datos = pd.concat(partes, ignore_index=True)
    return datos

def medir_memoria(df):
    """Calcula el uso de memoria de un DataFrame en megabytes (MB).
    
    Args:
        df (pd.DataFrame): DataFrame para el cual calcular el uso de memoria.
    
    Returns:
        float: Uso de memoria en megabytes (MB).
    
    """
    mb = df.memory_usage(deep=True).sum() / 1024**2
    return round(mb, 4)

# Sección de funciones para limpieza de datos

def eliminar_duplicados(df, subset=None):
    """Elimina filas duplicadas de un DataFrame.
    
    Args:
        df (pd.DataFrame): DataFrame del cual eliminar duplicados.
        subset (list, optional): Lista de columnas a considerar para identificar duplicados. 
                                 Si es None, se consideran todas las columnas.
    
    Returns:
        pd.DataFrame: DataFrame sin filas duplicadas.
    
    """
    return df.drop_duplicates(subset=subset)

def normalizar_sexo(df, columna='sexo', mapa_sexo=None):
    """Normaliza la columna de sexo a valores estandarizados 'M' y 'F'.
    
    Args:
        df (pd.DataFrame): DataFrame que contiene la columna de sexo.
        columna (str): Nombre de la columna de sexo a normalizar.
    
    Returns:
        pd.DataFrame: DataFrame con la columna de sexo normalizada.
    
    """    
    
    if mapa_sexo is None:
        mapa_sexo = {
            'M': 'M', 'MASCULINO': 'M', 'HOMBRE': 'M', 'MASC': 'M',
            'F': 'F', 'FEMENINO': 'F', 'MUJER': 'F', 'FEM': 'F',
        }
    
    df[columna] = df[columna].str.strip().str.upper().map(mapa_sexo)
    return df

def convertir_fechas(df, columnas_fechas):
    """Convierte columnas de fechas de formato string a datetime.
    
    Args:
        df (pd.DataFrame): DataFrame que contiene las columnas de fechas.
        columnas_fechas (list): Lista de nombres de columnas que contienen fechas.
    
    Returns:
        pd.DataFrame: DataFrame con las columnas de fechas convertidas a datetime.
    
    """
    for col in columnas_fechas:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    return df

In [72]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 1: INGESTA INTELIGENTE
# ══════════════════════════════════════════════════════════════════════════════
# Carga cartera, siniestros y catalogo de ramos/agentes.
# Usa SOLO las columnas que necesitas — justifica con comentario.
# Mide la memoria ahorrada vs cargar todo.

# Tu codigo aqui:

# Especificamos las rutas de los archivos a cargar.
ruta_cartera = 'datos/cartera_polizas.csv'
ruta_siniestros = 'datos/siniestros.csv'
ruta_catalogo_ramos = 'datos/catalogo_ramos.csv'
ruta_catalogo_agentes = 'datos/catalogo_agentes.csv'

# Obtenemos la lista de columnas analiticas para cada dataset
columnas_cartera = ['id_poliza','num_poliza','ramo','plan','status_poliza', # Caracteristicas basicas de la poliza
                                'nombre','apellido_paterno','apellido_materno', # Datos personales del asegurado
                                'rfc','edad','sexo','estado_civil','ocupacion', # Datos demograficos del asegurado
                                'fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia', 'fecha_nacimiento', # Fechas importantes para calcular antiguedad, edad, etc
                                'num_renovaciones','motivo_baja', # Datos de renovacion y baja
                                'suma_asegurada','deducible','prima_neta','prima_total', # Datos financieros de la poliza
                                'forma_pago','agente_id','canal_venta', # Datos de venta y distribucion
                                'estado','municipio','codigo_postal', # Datos geograficos
                                'marca_vehiculo','modelo_vehiculo','tipo_vehiculo',  # Datos de vehiculo para polizas de autos
                            ]

columnas_siniestros = ['id_siniestro','id_poliza', # Identificadores para relacionar con la cartera
                       'fecha_ocurrencia','fecha_reporte','fecha_apertura','fecha_ultimo_movimiento','fecha_cierre', # Fechas para calcular tiempos de resolucion
                       'ramo','tipo_siniestro','subtipo', 'descripcion_diagnostico', 'hospital_id', # Caracteristicas del siniestro
                       'monto_reclamado','monto_deducible_aplic','monto_coaseguro','monto_pagado','monto_reservado', 'moneda', # Datos financieros del siniestro
                       'status_siniestro','motivo_rechazo','id_ajustador' # Datos de gestion del siniestro
                       ]

columnas_catalogo_ramos = ['ramo', 'nombre_largo', # Identificador y nombre del ramo
                           'tasa_base' # Caracteristicas del ramo para analisis y filtrado
                           ]

columnas_catalogo_agentes = ['agente_id','nombre', 'region', # Datos basicos del agente para analisis de distribucion
                             'comision_pct']

columnas_datasets = {
    'cartera_polizas': columnas_cartera,
    'siniestros': columnas_siniestros,
    'catalogo_ramos': columnas_catalogo_ramos,
    'catalogo_agentes': columnas_catalogo_agentes,
}


# Cargamos cada dataset en el ciclo for

datos =[]
for k, v in columnas_datasets.items():
    df = cargar_datos(ruta_csv=f'datos/{k}.csv', columnas_analiticas=v)
    memoria = medir_memoria(df)
    memoria_completa = medir_memoria(cargar_datos(f'datos/{k}.csv'))  # Carga completa para comparar
    reduccion = f'{round((1 - memoria/memoria_completa)*100,1)}%'
    datos.append(dict(Dataset=k, 
                      Memoria_Completa_MB=memoria_completa, 
                      Memoria_Optimizada_MB=memoria, 
                      Reduccion_MB = reduccion))
    
df_comparativo = pd.DataFrame(datos)

print(df_comparativo.to_string(index=False))

         Dataset  Memoria_Completa_MB  Memoria_Optimizada_MB Reduccion_MB
 cartera_polizas             109.4458                73.7018        32.7%
      siniestros              26.6597                17.8385        33.1%
  catalogo_ramos               0.0010                 0.0007        30.0%
catalogo_agentes               0.0325                 0.0158        51.4%


In [73]:
# Haremos funciones para cada una de las tareas del proceso de limpieza, para que el codigo sea mas modular y facil de mantener.

def eliminar_duplicados(df, subset=None):
    """Elimina filas duplicadas de un DataFrame.
    
    Args:
        df (pd.DataFrame): DataFrame del cual eliminar duplicados.
        subset (list, optional): Lista de columnas a considerar para identificar duplicados. 
                                 Si es None, se consideran todas las columnas.
    
    Returns:
        pd.DataFrame: DataFrame sin filas duplicadas.
    
    """
    n_eliminados = df.duplicated(subset=subset).sum()
    return df.drop_duplicates(subset=subset), n_eliminados

def normalizar_sexo(df, columna='sexo', mapa_sexo=None):
    """Normaliza la columna de sexo a valores estandarizados 'M' y 'F'.
    
    Args:
        df (pd.DataFrame): DataFrame que contiene la columna de sexo.
        columna (str): Nombre de la columna de sexo a normalizar.
    
    Returns:
        pd.DataFrame: DataFrame con la columna de sexo normalizada.
    
    """    
    
    if mapa_sexo is None:
        mapa_sexo = {
            'M': 'M', 'MASCULINO': 'M', 'HOMBRE': 'M', 'MASC': 'M',
            'F': 'F', 'FEMENINO': 'F', 'MUJER': 'F', 'FEM': 'F',
        }
    
    df[columna] = df[columna].str.strip().str.upper().map(mapa_sexo)
    return df

def convertir_fechas(df, columnas_fechas):
    """Convierte columnas de fechas de formato string a datetime.
    
    Args:
        df (pd.DataFrame): DataFrame que contiene las columnas de fechas.
        columnas_fechas (list): Lista de nombres de columnas que contienen fechas.
    
    Returns:
        pd.DataFrame: DataFrame con las columnas de fechas convertidas a datetime.
    
    """
    for col in columnas_fechas:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    return df

def imputar_mediana_prima_neta_ramo(df, columna_imputacion='prima_neta', columna_grupo='ramo'):
    """Imputa valores faltantes en la columna de prima_neta usando la mediana por ramo.
    
    Args:
        df (pd.DataFrame): DataFrame que contiene las columnas a imputar y agrupar.
        columna_imputacion (str): Nombre de la columna con valores faltantes a imputar.
        columna_grupo (str): Nombre de la columna por la cual agrupar para calcular la mediana.
    
    Returns:
        pd.DataFrame: DataFrame con los valores imputados en la columna de prima_neta.
    
    """
    df[columna_imputacion] = df.groupby(columna_grupo)[columna_imputacion].transform(lambda x: x.fillna(x.median()))
    return df

def limpieza_codigo_postal(df, columna='codigo_postal', valor_relleno="NaN"):
    """Rellena valores faltantes en la columna de codigo_postal con un valor especifico.
    
    Args:
        df (pd.DataFrame): DataFrame que contiene la columna de codigo postal.
        columna (str): Nombre de la columna de codigo postal a limpiar.
        valor_relleno (str): Valor con el cual rellenar los valores faltantes.
    
    Returns:
        pd.DataFrame: DataFrame con los valores faltantes en codigo_postal rellenados.
    
    """
    df[columna] = df[columna].fillna(valor_relleno)
    return df

def optimizacion_categorias(df, columnas_category):
    """Convierte columnas con pocos valores unicos a tipo category para optimizar memoria.
    
    Args:
        df (pd.DataFrame): DataFrame que contiene las columnas a convertir.
        columnas_category (list): Lista de nombres de columnas a convertir a category.
    
    Returns:
        pd.DataFrame: DataFrame con las columnas convertidas a category.
    
    """
    for col in columnas_category:
        if col in df.columns:
            n_uniq = df[col].nunique()
            n_tot  = len(df)
            pct = n_uniq/n_tot
            print(f'  {col:<25}: {n_uniq:>5} unicos ({pct:.1%}) → category')
            df[col] = df[col].astype('category')
    return df

In [74]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 2: LIMPIEZA COMPLETA
# ══════════════════════════════════════════════════════════════════════════════
# 2a. Elimina duplicados de cartera
# 2b. Normaliza sexo con str.strip().str.upper() + .map(MAPA_SEXO)
# 2c. Convierte TODAS las fechas a datetime con errors='coerce'
# 2d. Rellena NaN de prima_neta con la MEDIANA POR RAMO (no global)
#     df.groupby('ramo')['prima_neta'].transform(lambda x: x.fillna(x.median()))
# 2e. Limpia codigo_postal (reemplaza 'N/D' con NaN)
# 2f. Aplica optimizacion de categoricas (sin tocar float64 de primas)

# Tu codigo aqui:

# Eliminamos duplicados del dataset de cartera
cartera_polizas = cargar_datos(ruta_csv=ruta_cartera, columnas_analiticas=columnas_cartera)
cartera_polizas, n_duplicados = eliminar_duplicados(cartera_polizas, subset=['id_poliza'])
print(f'Polizas duplicadas eliminadas: {n_duplicados} de un total de {len(cartera_polizas)+n_duplicados}')

# Normalizamos la columna de sexo
mapa_sexo = {
    'MASCULINO': 'M', 'FEMENINO': 'F', 'M':'M', 'F':'F'
}

print("-"*100)
print('Distribucion de sexo antes de normalizar:')
sexo_antes = cartera_polizas['sexo'].value_counts(dropna=False)
print(sexo_antes.to_string())

print()
cartera_polizas = normalizar_sexo(cartera_polizas, columna='sexo', mapa_sexo=mapa_sexo)
sexo_normalizado = cartera_polizas['sexo'].value_counts(dropna=False)
print('Distribucion de sexo despues de normalizar:')
print(sexo_normalizado.to_string())

print("-"*100)

# Convertimos las columnas de fechas a formato datetime
cols_fechas = ['fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia']

fechas_antes = cartera_polizas[cols_fechas]
print('Fechas antes de convertir:')
print(fechas_antes.dtypes)
print()

print("Fechas después de convertir:")
cartera_polizas = convertir_fechas(cartera_polizas, columnas_fechas=cols_fechas)
cartera_polizas["fecha_nacimiento"] = pd.to_datetime(cartera_polizas["fecha_nacimiento"], format="%d/%m/%Y", errors='coerce') # Convertimos la columna de fecha de nacimiento a formato datetime
fechas_despues = cartera_polizas[cols_fechas]
print(fechas_despues.dtypes)
print("-"*100)

# Imputamos los valores faltantes en prima_neta usando la mediana por ramo
n_nan_antes = cartera_polizas['prima_neta'].isna().sum()
print(f'Valores NaN en prima_neta antes de imputar: {n_nan_antes}')
cartera_polizas = imputar_mediana_prima_neta_ramo(cartera_polizas, columna_imputacion='prima_neta', columna_grupo='ramo')
n_nan_despues = cartera_polizas['prima_neta'].isna().sum()
print(f'Valores NaN en prima_neta despues de imputar: {n_nan_despues}')
print("-"*100)

# Limpiamos la columna de codigo_postal rellenando los NaN con "NaN"
n_nan_cp_antes = cartera_polizas['codigo_postal'].isna().sum()
print(f'Valores NaN en codigo_postal antes de limpiar: {n_nan_cp_antes}')
cartera_polizas = limpieza_codigo_postal(cartera_polizas, columna='codigo_postal', valor_relleno=0)
n_nan_cp_despues = cartera_polizas['codigo_postal'].isna().sum()
print(f'Valores NaN en codigo_postal despues de limpiar: {n_nan_cp_despues}')
print("-"*100)

# Aplicamos optimizacion de categoricas a las columnas especificadas
columnas_category = ['ramo','plan','canal_venta','forma_pago','estado','sexo','tipo_vehiculo']

cartera_polizas = optimizacion_categorias(cartera_polizas, columnas_category=columnas_category)

Polizas duplicadas eliminadas: 0 de un total de 50000
----------------------------------------------------------------------------------------------------
Distribucion de sexo antes de normalizar:
sexo
M            16697
F            16585
Femenino      8466
Masculino     8252

Distribucion de sexo despues de normalizar:
sexo
F    25051
M    24949
----------------------------------------------------------------------------------------------------
Fechas antes de convertir:
fecha_emision            object
fecha_inicio_vigencia    object
fecha_fin_vigencia       object
dtype: object

Fechas después de convertir:
fecha_emision            datetime64[ns]
fecha_inicio_vigencia    datetime64[ns]
fecha_fin_vigencia       datetime64[ns]
dtype: object
----------------------------------------------------------------------------------------------------
Valores NaN en prima_neta antes de imputar: 1000
Valores NaN en prima_neta despues de imputar: 0
--------------------------------------------------

In [75]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 3: ENRIQUECIMIENTO
# ══════════════════════════════════════════════════════════════════════════════
# 3a. Merge con catalogo_ramos: agregar nombre_largo, tasa_base
# 3b. Merge con catalogo_agentes: agregar nombre del agente, region
# 3c. Crear: g_edad con pd.cut
# 3d. Crear: prima_calc = suma_asegurada * tasa_base * 1.16
# 3e. Crear: nivel_riesgo con .apply(clasificar_riesgo) — de mi_modulo
# 3f. Crear: edad_calc desde fecha_nacimiento
# 3g. Crear: dias_vigencia, fraccion_expuesta

# Tu codigo aqui:
import mi_modulo

# Joins con catalogos para enriquecer la cartera
catalogo_ramos = cargar_datos(ruta_csv=ruta_catalogo_ramos, columnas_analiticas=columnas_catalogo_ramos)
catalogo_agentes = cargar_datos(ruta_csv=ruta_catalogo_agentes, columnas_analiticas=columnas_catalogo_agentes)

cartera_polizas_ramo = pd.merge(cartera_polizas, catalogo_ramos, left_on='ramo', right_on='ramo', how='left', suffixes=('', '_ramos'))
cartera_polizas_ramo_agente = pd.merge(cartera_polizas_ramo, catalogo_agentes, left_on='agente_id', right_on='agente_id', how='left', suffixes=('', '_agentes'))

# Columnas adicionales para analisis
cartera_polizas_ramo_agente['g_edad'] = pd.cut(cartera_polizas_ramo_agente['edad'], bins=[0,30,45,60,100],
                           labels=['18-30','31-45','46-60','61+'])

cartera_polizas_ramo_agente['prima_calc'] = cartera_polizas_ramo_agente['suma_asegurada'] * cartera_polizas_ramo_agente['tasa_base'] * 1.16

cartera_polizas_ramo_agente['comision_agente'] = cartera_polizas_ramo_agente['prima_total'] * (cartera_polizas_ramo_agente['comision_pct']/100)

# Para poder clasificar el nivel de riesgo, necesitamos los siniestros asociados a cada poliza. Haremos un merge con el dataset de siniestros para obtener esa informacion.
siniestros = cargar_datos(ruta_csv=ruta_siniestros, columnas_analiticas=columnas_siniestros)
siniestros_agg = siniestros.groupby('id_poliza').agg(
    num_siniestros = ('id_siniestro','count'),
    monto_total_reclamado = ('monto_reclamado','sum')
).reset_index()

cartera_polizas_ramo_agente_siniestros = pd.merge(cartera_polizas_ramo_agente, siniestros_agg, left_on='id_poliza', right_on='id_poliza', how='left', suffixes=('', '_siniestros'))

cartera_polizas_ramo_agente_siniestros['num_siniestros'] = cartera_polizas_ramo_agente_siniestros['num_siniestros'].fillna(0)
cartera_polizas_ramo_agente_siniestros['monto_total_reclamado'] = cartera_polizas_ramo_agente_siniestros['monto_total_reclamado'].fillna(0)

cartera_polizas_ramo_agente_siniestros['nivel_riesgo'] = cartera_polizas_ramo_agente_siniestros.apply(
    lambda x: mi_modulo.clasificar_riesgo(x['num_siniestros']), axis=1
)


# Ahora calculamos la edad a partir de la fecha de nacimiento, los dias de vigencia y la fraccion expuesta a riesgo, usando las fechas ya convertidas a datetime.
hoy = pd.Timestamp.today()

cartera_polizas_ramo_agente_siniestros['edad_calc'] = ((hoy - cartera_polizas_ramo_agente_siniestros['fecha_nacimiento']).dt.days / 365.25).round(1)

cartera_polizas_ramo_agente_siniestros['dias_vigencia'] = (cartera_polizas_ramo_agente_siniestros['fecha_fin_vigencia'] - cartera_polizas_ramo_agente_siniestros['fecha_inicio_vigencia']).dt.days

cartera_polizas_ramo_agente_siniestros['fraccion_expuesta'] = ( (hoy - cartera_polizas_ramo_agente_siniestros['fecha_inicio_vigencia']).dt.days / cartera_polizas_ramo_agente_siniestros['dias_vigencia'] ).clip(0, 1).round(4)

# Finalmente, asignamos la cartera enriquecida a una variable con un nombre representativo.
cartera_completa = cartera_polizas_ramo_agente_siniestros
print(f'Cartera enriquecida lista con {cartera_completa.shape[0]:,} filas y {cartera_completa.shape[1]} columnas')

Cartera enriquecida lista con 50,000 filas y 46 columnas


In [76]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 4: ANALISIS Y REPORTES
# ══════════════════════════════════════════════════════════════════════════════
# 4a. groupby+agg por ramo: polizas, prima_total, prima_prom, pct_cartera
# 4b. groupby+agg por agente: polizas, prima_total, comision (10%)
# 4c. pivot_table prima por ramo x g_edad con margins=True
# 4d. pivot_table polizas por estado x ramo
# 4e. Identifica: ramo con mayor prima total y zona con mayor frecuencia

# Tu codigo aqui:

# Resumen por ramo
resumen_ramo = cartera_completa.groupby("ramo").agg(
    polizas = ('id_poliza','count'),
    prima_total = ('prima_total','sum'),
    prima_prom = ('prima_total','mean'),
    pct_cartera = ('prima_total', lambda x: x.sum()/cartera_completa['prima_total'].sum())
).reset_index()

# Resumen por agente
resumen_agente = cartera_completa.groupby("nombre_agentes").agg(
    polizas = ('id_poliza','count'),
    prima_total = ('prima_total','sum'),
    comision = ('comision_agente', lambda x: x.sum())
).reset_index()

# Pivot table prima por ramo x grupo de edad
tabla_prima = pd.pivot_table(
    cartera_completa,
    values = 'prima_total',
    index = 'ramo',
    columns = 'g_edad',
    aggfunc = 'sum',
    fill_value = 0,
    margins = True,
    margins_name = 'TOTAL',
    observed=False).reset_index()

# Pivot table polizas por estado x ramo
tabla_estado_ramo = pd.pivot_table(
    cartera_completa,
    values = 'id_poliza',
    index = 'estado',
    columns = 'ramo',
    aggfunc = 'count',
    fill_value = 0,
    margins = True,
    margins_name = 'TOTAL',
    observed=False).reset_index()

# Ramo con mayor prima total
ramo_mayor_prima = resumen_ramo.sort_values('prima_total', ascending=False).iloc[0]['ramo']
print(f'Ramo con mayor prima total: {ramo_mayor_prima}')

# Zona con mayor frecuencia (número de siniestros)
tabla_estado_ramo_siniestros = pd.pivot_table(
    cartera_completa,
    values = 'num_siniestros',
    index = 'estado',
    columns = 'ramo',
    aggfunc = 'sum',
    fill_value = 0,
    margins = True,
    margins_name = 'TOTAL',
    observed=False).reset_index()

zona_mayor_frecuencia = tabla_estado_ramo_siniestros[tabla_estado_ramo_siniestros["estado"] != "TOTAL"].sort_values('TOTAL', ascending=False).iloc[0]['estado'] # Quitamos la fila de TOTAL para identificar la zona con mayor frecuencia de siniestros
print(f'Zona con mayor frecuencia de siniestros: {zona_mayor_frecuencia}')

Ramo con mayor prima total: GMM
Zona con mayor frecuencia de siniestros: Queretaro


In [77]:
from openpyxl.styles import PatternFill
from openpyxl.styles import Font
from openpyxl.utils import get_column_letter

# ══════════════════════════════════════════════════════════════════════════════
# FASE 5: EXPORTAR
# ══════════════════════════════════════════════════════════════════════════════
# Excel con 5 hojas: Cartera_Limpia, Resumen_Ramo, Resumen_Agente,
#                    Pivot_Prima, Pivot_Zona
# Parquet: cartera_q1_2026_final.parquet
# Compara tamano CSV equivalente vs Parquet

# Tu codigo aqui:
# ── Guardar en todos los formatos y comparar ──────────────────────────────────

# Exportamos a Excel con varias hojas
header_fill = PatternFill(start_color='002060', end_color='002060', fill_type='solid')
header_font = Font(color='FFFFFF', bold=True)

sheets = [
    ('Cartera_Limpia', cartera_completa),
    ('Resumen_Ramo', resumen_ramo),
    ('Resumen_Agente', resumen_agente),
    ('Pivot_Prima', tabla_prima),
    ('Pivot_Zona', tabla_estado_ramo_siniestros),
]

with pd.ExcelWriter('datos/reporte_final.xlsx', engine='openpyxl') as writer:
    for sheet_name, df_export in sheets:
        df_export.to_excel(writer, sheet_name=sheet_name, index=False)

    for sheet_name, df_export in sheets:
        ws = writer.sheets[sheet_name]
        ws.sheet_view.showGridLines = False

        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font

        for idx, col in enumerate(df_export.columns, start=1):
            max_len = len(str(col))
            for cell in ws[get_column_letter(idx)]:
                if cell.value is not None:
                    max_len = max(max_len, len(str(cell.value)))
            ws.column_dimensions[get_column_letter(idx)].width = max_len + 2

# Exportamos la cartera completa a formato Parquet
cartera_completa.to_parquet('datos/cartera_q1_2026_final.parquet', index=False)

# Exportamos la cartera completa a formato CSV para comparar tamaños
cartera_completa.to_csv('datos/cartera_q1_2026_final.csv', index=False)

# Comparamos tamaños de los archivos CSV y Parquet
size_csv = os.path.getsize('datos/cartera_q1_2026_final.csv') / 1024**2
size_parquet = os.path.getsize('datos/cartera_q1_2026_final.parquet') / 1024**2
print(f'Tamaño CSV: {size_csv:.2f} MB')
print(f'Tamaño Parquet: {size_parquet:.2f} MB')
print(f'Parquet es {(1 - size_parquet/size_csv)*100:.1f}% más pequeño que CSV')

Tamaño CSV: 17.02 MB
Tamaño Parquet: 3.44 MB
Parquet es 79.8% más pequeño que CSV


In [78]:
# ── Solucion resumida (descomenta solo si necesitas referencia) ──────────────

# FASE 1:
# df = pd.read_csv('datos/cartera_polizas.csv', usecols=ANALITICAS, na_values=['N/D','N/A'])
# ramos_cat = pd.read_csv('datos/catalogo_ramos.csv')
# agentes_cat = pd.read_csv('datos/catalogo_agentes.csv')

# FASE 2:
# df = df.drop_duplicates()
# df['sexo'] = df['sexo'].str.strip().str.upper().map(MAPA_SEXO)
# for col in ['fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia']:
#     df[col] = pd.to_datetime(df[col], errors='coerce')
# df['fecha_nacimiento'] = pd.to_datetime(df['fecha_nacimiento'],format='%d/%m/%Y',errors='coerce')
# df['prima_neta'] = df.groupby('ramo')['prima_neta'].transform(lambda x: x.fillna(x.median()))
# df['codigo_postal'] = df['codigo_postal'].replace('N/D', np.nan)
# for col in ['ramo','plan','canal_venta','forma_pago','estado','sexo','tipo_vehiculo']:
#     if col in df.columns: df[col] = df[col].astype('category')

# FASE 3 (extracto):
# df = pd.merge(df, ramos_cat[['ramo','nombre_largo','tasa_base']], on='ramo', how='left')
# df['g_edad'] = pd.cut(df['edad'],bins=[0,30,45,60,100],labels=['18-30','31-45','46-60','61+'])
# df['prima_calc'] = df['suma_asegurada'] * df['tasa_base'] * 1.16
# from mi_modulo import clasificar_riesgo
# df['nivel_riesgo'] = df['prima_calc'].apply(lambda p: 'ALTO' if p>15000 else 'MEDIO' if p>6000 else 'BAJO')

# FASE 5:
# with pd.ExcelWriter('datos/reporte_ejecutivo_Q1_2026.xlsx', engine='openpyxl') as w:
#     df.to_excel(w,'Cartera_Limpia',index=False)
#     resumen_ramo.to_excel(w,'Resumen_Ramo',index=False)
#     resumen_agente.to_excel(w,'Resumen_Agente',index=False)
#     tabla_prima.to_excel(w,'Pivot_Prima')
#     tabla_zona.to_excel(w,'Pivot_Zona')
# df.to_parquet('datos/cartera_q1_2026_final.parquet', index=False)
# print(f'CSV: {df.to_csv(index=False).encode().__len__()/1024:.0f} KB')
# print(f'Parquet: {os.path.getsize("datos/cartera_q1_2026_final.parquet")/1024:.0f} KB')

---
## Resumen: Lo que Aprendiste en la Sesion 8

| Duda | Herramienta | Aprendizaje clave |
|------|-------------|-------------------|
| 46 columnas | `usecols` + taxonomia | Clasificar antes de cargar — decision de negocio |
| Texto sucio | `str.strip/upper/map` | Normalizar ANTES del primer groupby |
| Fechas texto | `pd.to_datetime(errors='coerce')` | Nunca detener el pipeline por fechas invalidas |
| JSON anidado | `json_normalize(sep='_')` | Aplanar antes de analizar |
| 90k filas | `chunksize` + `category` | Medir memoria antes de decidir |
| Downcast riesgoso | Regla float32 < 100k MXN | float64 para montos grandes siempre |
| Polars | `pl.scan_csv().collect()` | Lazy evaluation = optimizacion automatica |

**T5 Pandas — COMPLETADO**

**Proxima sesion — Mie 6 mayo, 18:00 h:**
T6 Visualizacion — Matplotlib, Seaborn y Plotly.

```bash
git add sesion8_M1_notebook.ipynb
git commit -m "Sesion 8: pipeline completo datos reales - str datetime JSON Polars"
git push
```

---
*Diplomado ML en Seguros · FC UNAM · 2026*